# FungMod from zero to a complete virtual-experiment report

This notebook starts at the installed public API and ends with a
self-describing output bundle: preflight decisions, trajectories,
final metrics, threshold times, uncertainty summaries, provenance,
limitations, suggested experiments, quick-look figures, and a
Markdown/HTML report.

**Scope:** the bundled cellulose-equivalent enzyme-chain case is an
exploratory software-verified example. It is not whole-fungus
physiology, organism-specific evidence, calibration, or empirical
validation. Runtime environment-grid values remain metadata unless an
explicit response law or condition-specific record is active.

**Validation:** Software execution is not empirical validation.


## Install

In a fresh Python 3.11+ environment:

```bash
python -m pip install fungmod
```

The distribution provides both `import fungmod` and the original
`import fungal_model` namespace.


In [ ]:
import json
import os
from pathlib import Path

import fungmod as fm

OUTPUT_ROOT = Path(
    os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks")
).resolve()
OUTPUT = OUTPUT_ROOT / "20_zero_to_complete_virtual_experiment"
SAMPLE_COUNT = int(os.environ.get("FUNGMOD_NOTEBOOK_SAMPLES", "8"))
OUTPUT.mkdir(parents=True, exist_ok=True)

{"fungmod_version": fm.__version__, "output_directory": str(OUTPUT)}


## 1. Define a screen from researcher-facing names

The grid creates four explicit environment cases. The public resolver
maps the fungus/source and substrate aliases to registry IDs. Creating
the study does not run a model.


In [ ]:
grid = fm.environment_grid(
    temperature_C=[25.0, 30.0],
    ph=[4.5, 5.5],
    oxygen="aerobic",
)
study = fm.virtual_experiment(
    fungi="generic cellulase source",
    substrates="cellulose film",
    environments=grid,
)

study.to_dict()


## 2. Run modelability preflight before simulation

Preflight is a guardrail, not the scientific result. It makes missing,
exploratory, incompatible, and unsupported inputs visible before the
solver runs.


In [ ]:
preflight = study.preflight(mode="exploratory")
preflight_rows = [report.to_dict() for report in preflight]
{
    "case_count": study.case_count,
    "statuses": [row["status"] for row in preflight_rows],
    "summaries": [report.summary() for report in preflight],
}


## 3. Simulate the allowed exploratory cases

A fixed seed makes sampled exploratory ranges reproducible. Quick-look
plots are presentation aids generated from exported tables; they are
not validation plots.


In [ ]:
result = study.simulate(
    mode="exploratory",
    n_samples=SAMPLE_COUNT,
    seed=20260730,
    output_dir=OUTPUT,
    quicklook=True,
)

{
    "mode": result.mode,
    "samples_per_case": result.n_samples,
    "case_results": len(result.screen_result.case_results),
    "quicklook_files": [Path(path).name for path in result.quicklook_paths],
}


## 4. Inspect mechanism, assumptions, and numerical results together

Read mechanism and assumption rows before interpreting curves. The
first few rows below are deliberately kept as dictionaries so units,
status labels, and allowed-use fields remain visible.


In [ ]:
mechanisms = result.mechanism_summary()
assumptions = result.assumption_summary()
final_metrics = result.final_metrics()
threshold_times = result.threshold_times()

{
    "mechanisms": mechanisms[:4],
    "assumptions": assumptions[:4],
    "final_metrics": final_metrics[:6],
    "threshold_times": threshold_times[:6],
}


## 5. Inspect uncertainty and comparison guardrails

Quantile bands summarize explicit sampled inputs and resulting
trajectories. They are not posterior intervals or empirical confidence
intervals. Comparison rows carry their own `comparison_allowed`,
`ranking_allowed`, and blocking-reason fields.


In [ ]:
uncertainty_rows = result.uncertainty_summary()
trajectory_rows = result.trajectory_quantiles()
comparison_rows = result.comparison_summary()

{
    "uncertainty_examples": uncertainty_rows[:5],
    "trajectory_quantile_examples": trajectory_rows[:5],
    "comparison_guardrails": [
        {
            key: row.get(key, "")
            for key in (
                "source_metric",
                "comparison_allowed",
                "ranking_allowed",
                "ranking_blocking_reason",
            )
        }
        for row in comparison_rows[:6]
    ],
}


## 6. Let the result identify limitations and next measurements

FungMod keeps unsupported biology and missing evidence in standard
tables rather than hiding them behind a successful solver run.


In [ ]:
limitations = result.limitations()
suggestions = result.suggested_experiments()
provenance = result.provenance()

{
    "limitations": limitations[:8],
    "suggested_experiments": suggestions[:8],
    "provenance": provenance[:6],
}


## 7. Write a navigable report and verify the manifest

The report is rendered only from existing standard tables. The manifest
is the machine-readable inventory for downstream analysis and archiving.


In [ ]:
report_path = result.write_report(
    OUTPUT / "report",
    include_html=True,
    include_index=True,
)
manifest = json.loads((OUTPUT / "output_manifest.json").read_text(encoding="utf-8"))
required = {
    "time_series_long.csv",
    "final_metrics.csv",
    "threshold_times.csv",
    "uncertainty_summary.csv",
    "trajectory_quantiles.csv",
    "provenance_table.csv",
    "limitations_table.csv",
    "suggested_experiments.csv",
    "report/virtual_experiment_report.md",
    "report/virtual_experiment_report.html",
    "report/index.html",
}
missing = sorted(required - set(manifest["files"]))
assert not missing, missing

{
    "report": str(report_path),
    "manifest_schema": manifest["output_schema_version"],
    "artifact_count": len(manifest["files"]),
    "missing_required_artifacts": missing,
}


## What the notebook established

- The installed package can resolve its bundled registry without a
  repository checkout.
- Modelability, mechanisms, assumptions, numerical outputs,
  uncertainty, provenance, and limitations stay connected.
- Every displayed result is recoverable from exported tables.

It did **not** establish organism-specific performance, experimental
agreement, publication-grade calibration, or whole-fungus behavior.
